# Routing BENCHMARK — Gemma 4 E4B chess-coach (for the report)

Runs the **serious** routing benchmark on Kaggle (T4, up to 12h — no Colab disconnect): the
**trained v4 adapter vs the untrained base Gemma** on the SAME harness, over a large stratified
sample of the validation set. Produces, for BOTH conditions: a confusion matrix +
precision/recall/F1 + exact-name accuracy + format validity + per-slice routing + throughput,
plus the **adapter-over-base lift** (the honest 'against others' — it isolates what the SFT
bought; the base reuses the same loaded model with the LoRA disabled, so no 2nd load / no OOM).

**Setup:** Settings → Accelerator → **GPU T4 x2** (one is enough). Add your **HF_TOKEN** under
Add-ons → Secrets. Run top to bottom; the report + PNGs land in `/kaggle/working/`.

In [ ]:
# Cell 1 - config
import os
BASE_REPO = 'unsloth/gemma-4-E4B-it'   # the base the adapter was trained on
REPO_URL  = 'https://github.com/RyanDev1st/llm-and-engine.git'
BRANCH    = 'feat/chess-coach-sft'
ADAPTER_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v4'
TAG = 'best'
WORKDIR   = '/kaggle/working/llm-and-engine'
BASE_DIR  = '/kaggle/working/gemma4_e4b'
ADAPTER_DIR = '/kaggle/working/adapter/best'
# Coverage knobs (Kaggle has hours, so go big):
PER_SLICE    = 0          # 0 = the FULL validation set (~2731 rows). Set e.g. 40 for a subset.
TIME_BUDGET  = 6 * 3600   # seconds PER condition (adapter, then base). 6h+6h fits a 12h session.
print('full val' if PER_SLICE == 0 else f'{PER_SLICE} rows/slice', '| budget/condition', TIME_BUDGET//3600, 'h')

In [ ]:
# Cell 2 - clone repo (prints HEAD so the commit is visible) + deps
import subprocess, sys
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True, env=env)
print('HEAD:', subprocess.run(['git','-C',WORKDIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','peft','bitsandbytes','sentencepiece','protobuf','python-chess'], check=True)
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
import transformers, peft, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# Cell 3 - download the E4B base (~9GB) + the v4 adapter
from huggingface_hub import login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    login(os.environ['HF_TOKEN'])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=['*.json','*.safetensors','*.model','*.txt','tokenizer*'])
snapshot_download(repo_id=ADAPTER_REPO, local_dir='/kaggle/working/adapter', allow_patterns=[f'{TAG}/*'])
need = ('adapter_model.safetensors', 'adapter_config.json')
assert all(os.path.exists(f'{ADAPTER_DIR}/{f}') for f in need), f'adapter not at {ADAPTER_DIR}'
print('base + adapter ready:', os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 4 - RUN THE BENCHMARK (adapter vs base; hours-safe; writes the report + PNGs)
# Loads the model ONCE; the base condition reuses it with the LoRA disabled (no 2nd load, no OOM).
# Saves docs/findings/<date>-routing-benchmark.md + -adapter.png + -base.png; copied to /kaggle/working.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
       '--per-slice', str(PER_SLICE), '--time-budget', str(TIME_BUDGET)]
print('running:', ' '.join(cmd), '\n', flush=True)
subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env={**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}, check=True)
# surface the artifacts in the Kaggle output pane + show the matrices inline
for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark*'):
    shutil.copy(f, '/kaggle/working/')
from IPython.display import Image, Markdown, display
md = sorted(glob.glob('/kaggle/working/*-routing-benchmark.md'))
if md: display(Markdown(open(md[-1]).read()))
for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-*.png')):
    print(png); display(Image(filename=png))